# ReelAnimator — Cloud Pipeline (Google Colab)

Runs the full ReelAnimator pipeline in Colab with GPU:
**Whisper transcribe → script + translate → Sarvam Tamil TTS → hyper-motion → b-roll (raw footage) → HyperFrames compose → earshot SFX → ffmpeg mux**.

**Setup:** Runtime → Change runtime type → T4 GPU (or any GPU).

**Before running:** Set `SARVAM_API_KEY` and `GITHUB_TOKEN` (if repo is private) in the **Secrets** panel (left sidebar 🔑).

## 1. Environment setup

In [ ]:
#@title Install Node 22, pnpm, ffmpeg, whisper deps
import os
os.environ['DEBIAN_FRONTEND'] = 'noninteractive'

# ffmpeg
!apt-get -qq install -y ffmpeg > /dev/null 2>&1
!ffmpeg -version | head -1

# Node 22 via nvm
!curl -fsSL https://raw.githubusercontent.com/nvm-sh/nvm/v0.40.1/install.sh | bash > /dev/null 2>&1
os.environ['NVM_DIR'] = '/root/.nvm'
!bash -c 'source /root/.nvm/nvm.sh && nvm install 22 && nvm alias default 22 && corepack enable && corepack prepare pnpm@latest --activate'
!bash -c 'source /root/.nvm/nvm.sh && nvm use 22 && node --version && pnpm --version'

In [ ]:
#@title Install Python deps (openai-whisper, yt-dlp) + uv + earshot
!pip install -q openai-whisper yt-dlp > /dev/null 2>&1
!curl -fsSL https://astral.sh/uv/install.sh | sh > /dev/null 2>&1
import os
os.environ['PATH'] = '/root/.local/bin:' + os.environ.get('PATH', '')
!uv tool install "earshot[mcp] @ git+https://github.com/Leonxlnx/earshot" > /dev/null 2>&1
!earshot doctor --fetch > /dev/null 2>&1 || true
print('Python + earshot ready')

In [ ]:
#@title Install hyper-motion CLI (built from source)
import os
os.environ['NVM_DIR'] = '/root/.nvm'
!bash -c 'source /root/.nvm/nvm.sh && nvm use 22 && cd /tmp && git clone --depth 1 https://github.com/psiddharthdesign/hypermotion.git && cd hypermotion/cli && pnpm install > /dev/null 2>&1 && pnpm run build > /dev/null 2>&1 && mkdir -p /root/.local/bin && ln -sf /tmp/hypermotion/cli/bin/hypermotion.mjs /root/.local/bin/hypermotion && ln -sf /tmp/hypermotion/cli/bin/hypermotion-mcp.mjs /root/.local/bin/hypermotion-mcp && chmod +x /tmp/hypermotion/cli/bin/*.mjs'
os.environ['PATH'] = '/root/.local/bin:' + os.environ.get('PATH', '')
!hypermotion --help 2>&1 | head -2 || echo 'hypermotion installed'

## 2. Clone the ReelAnimator repo

In [ ]:
#@title Clone repo (set GITHUB_REPO to your fork if private)
GITHUB_REPO = ''  #@param {type:'string'}  # e.g. 'youruser/ReelAnimator'. Leave blank to upload files manually.
GITHUB_TOKEN = ''  #@param {type:'string'}  # from Colab Secrets (for private repos)

if GITHUB_REPO:
    if GITHUB_TOKEN:
        !git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git /content/ReelAnimator
    else:
        !git clone https://github.com/{GITHUB_REPO}.git /content/ReelAnimator
    %cd /content/ReelAnimator
else:
    print('No repo specified. Upload your project zip or clone manually below.')
    print('Example: !git clone https://github.com/youruser/ReelAnimator.git /content/ReelAnimator')
    print('Then: %cd /content/ReelAnimator')

## 3. Configure API keys

In [ ]:
#@title Set Sarvam API key + config
import os
from google.colab import userdata

# Get Sarvam key from Colab Secrets (or paste here)
SARVAM_API_KEY = userdata.get('SARVAM_API_KEY')  # @param {type:'string'}

os.environ['SARVAM_API_KEY'] = SARVAM_API_KEY

# Write .env
env_content = f'''SARVAM_API_KEY={SARVAM_API_KEY}
SARVAM_VOICE_ID_TA=vijay
SARVAM_VOICE_ID_EN=shubh
TTS_PROVIDER=sarvam
WHISPER_MODEL=small
PYTHON_VENV=python3
KILO_MODEL=kilo/kilo-auto/free
'''
with open('/content/ReelAnimator/.env', 'w') as f:
    f.write(env_content)
print('SARVAM_API_KEY set:', SARVAM_API_KEY[:8] + '...' if SARVAM_API_KEY else 'MISSING — set it in Colab Secrets')

## 4. Upload raw footage + script

In [ ]:
#@title Upload raw footage (optional) or use a topic
from google.colab import files
import os

os.makedirs('/content/ReelAnimator/Raw Footage', exist_ok=True)
os.makedirs('/content/ReelAnimator/out/first_reel', exist_ok=True)

print('Upload your raw footage (mp4/mov) — or skip to use a topic instead:')
try:
    uploaded = files.upload()
    for fn, data in uploaded.items():
        with open(f'/content/ReelAnimator/Raw Footage/{fn}', 'wb') as f:
            f.write(data)
        print(f'Saved: Raw Footage/{fn}')
except Exception as e:
    print('Upload skipped:', e)

In [ ]:
#@title Create a bilingual script (EN|TA per line) — edit as needed
script = '''Tamil cinema is taking over the world|தமிழ் சினிமா உலகை கைப்பற்றுகிறது
From Rajinikanth to RRR the impact is massive|ரஜினிகாந்த் முதல் RRR வரை தாக்கம் மிகப்பெரியது
Streaming platforms are investing big in Tamil films|ஸ்ட்ரீமிங் தளங்கள் தமிழ் படங்களில் பெரிய முதலீடு செய்கின்றன
Dubbed Tamil movies are trending globally|டப் செய்யப்பட்ட தமிழ் படங்கள் உலகளவில் டிரெண்டிங்
The future of Tamil entertainment looks bright|தமிழ் பொழுதுபோக்கின் எதிர்காலம் பிரகாசமாக உள்ளது'''

with open('/content/ReelAnimator/out/first_reel/script_bilingual.txt', 'w') as f:
    f.write(script)
print('Script written. Edit the cell and re-run to customize.')

## 5. Install Node deps + run pipeline

In [ ]:
#@title pnpm install + run the pipeline
import os
os.environ['NVM_DIR'] = '/root/.nvm'
os.environ['PATH'] = '/root/.local/bin:/root/.nvm/versions/node/v22.*/bin:' + os.environ.get('PATH', '')

%cd /content/ReelAnimator
!bash -c 'source /root/.nvm/nvm.sh && nvm use 22 && corepack enable && pnpm install' 2>&1 | tail -5

In [ ]:
#@title Run: from-text (bilingual script, no LLM needed)
import os
os.environ['NVM_DIR'] = '/root/.nvm'

%cd /content/ReelAnimator
!bash -c 'source /root/.nvm/nvm.sh && nvm use 22 && export PATH="/root/.local/bin:$PATH" && pnpm reel from-text --script-file out/first_reel/script_bilingual.txt --bilingual --lang ta' 2>&1 | tail -40

In [ ]:
#@title (Alternative) Run: from-video (transcribe raw footage with GPU Whisper)
import os
os.environ['NVM_DIR'] = '/root/.nvm'

%%bash
source /root/.nvm/nvm.sh && nvm use 22
export PATH="/root/.local/bin:$PATH"
cd /content/ReelAnimator
# Extract audio from raw footage, transcribe with GPU whisper
RAW=$(ls "Raw Footage"/*.mp4 2>/dev/null | head -1)
if [ -z "$RAW" ]; then echo 'No raw footage found'; exit 0; fi
ffmpeg -nostdin -y -i "$RAW" -vn -ac 1 -ar 16000 out/first_reel/audio.wav 2>/dev/null
# Transcribe with GPU (whisper small)
whisper out/first_reel/audio.wav --model small --language ta --output_format json --output_dir out/first_reel --word_timestamps True --device cuda 2>&1 | tail -10

## 6. Download the result

In [ ]:
#@title Download the final reel
from google.colab import files
import glob, os

reels = glob.glob('/content/ReelAnimator/out/*/reel_ta.mp4')
if reels:
    latest = max(reels, key=os.path.getmtime)
    print(f'Downloading: {latest}')
    files.download(latest)
    # Also download beats.json + metadata
    j = os.path.join(os.path.dirname(latest), 'beats.json')
    if os.path.exists(j):
        files.download(j)
else:
    print('No reel found. Check the pipeline output above for errors.')

## Notes

- **GPU Whisper**: On Colab GPU, use `--device cuda` for fast transcription (the `from-video` cell shows this).
- **hyper-motion**: Needs the desktop app for `.hype` rendering; on Colab it falls back to solid backgrounds (same as local). The GSAP animations in `compose.ts` still run.
- **Sarvam TTS**: Uses `bulbul:v3` with Tamil voice `vijay` (`ta-IN`). Better Tamil quality than ElevenLabs.
- **Raw footage**: Used as full-bleed background for all beats (Ken Burns zoom via GSAP).
- **Translation**: Uses Kilo free model (`kilo/kilo-auto/free`) or skip with `--bilingual` script.
- **earshot SFX**: Whoosh/pop/impact on beat cuts (252-sound bank, no models needed for `sfx`).